# AAI-540 Feature Store — AWS Starter Notebook

Connects to the project data lake (`s3://jonno-lucas-steve-bucket/usd-aai540-group1/`)
and the Glue catalog (`sagemaker_featurestore`) and demonstrates the typical 
offline query pattern:

**Region:** `us-east-2` · **Workgroup:** `primary` ·
Athena results land at `s3://jonno-lucas-steve-bucket/usd-aai540-group1/athena-results/`.

## Setup

Most SageMaker images already have `awswrangler`, `boto3`, `pandas`,
`numpy`, `sklearn`, `matplotlib`, `seaborn` pre-installed. If you hit
`ImportError`, uncomment the install line below.

In [ ]:
# !pip install -q awswrangler

In [1]:
import boto3
import awswrangler as wr
import numpy as np
import pandas as pd


pd.options.display.float_format = "{:,.2f}".format

# Confirm which IAM identity this notebook is running as
sts = boto3.client("sts", region_name="us-east-2")
print(sts.get_caller_identity()["Arn"])

arn:aws:iam::541974874359:user/jonno


In [2]:
# ---- Constants ----
REGION     = "us-east-2"
BUCKET     = "jonno-lucas-steve-bucket"
PROJECT    = "usd-aai540-group1"
SILVER_DB  = "aai540_silver"
GOLD_DB    = "aai540_gold"
ATHENA_OUT = f"s3://{BUCKET}/{PROJECT}/athena-results/"

# awswrangler picks up region from the boto3 default session
boto3.setup_default_session(region_name=REGION)

## 1. Load the feature store database

The full table is small (~2,500 rows × ~20 columns), so we pull it all
into memory. — Athena charges by data scanned.

In [3]:
df = wr.athena.read_sql_query(
    "SELECT * from x_attendance_y_sales_1779629962",
    database="sagemaker_featurestore",
    s3_output=ATHENA_OUT,
)

print(f"shape:      {df.shape}")
print(f"rows w/ est_attendance:   {(df['total-est-attendance'] > 0).sum()}")
df.head()

shape:      (2554, 7)
rows w/ est_attendance:   829


,record-identifier,event-time,total-est-attendance,taxable-sales-usd,write_time,api_invocation_time,is_deleted
0,06023-2021Q4,2021-12-31T00:00:00Z,18360,632299660,2026-05-24 14:26:22.863,2026-05-24 14:22:37,False
1,06101-2023Q2,2023-06-30T00:00:00Z,0,644345533,2026-05-24 14:26:21.541,2026-05-24 14:22:33,False
2,06051-2017Q1,2017-03-31T00:00:00Z,0,82980358,2026-05-24 14:26:22.668,2026-05-24 14:22:11,False
3,06009-2017Q1,2017-03-31T00:00:00Z,0,87373772,2026-05-24 14:26:22.668,2026-05-24 14:22:18,False
4,06057-2017Q1,2017-03-31T00:00:00Z,0,300139165,2026-05-24 14:26:22.668,2026-05-24 14:22:18,False


In [4]:
# Schema + null check
pd.DataFrame({
    "dtype":    df.dtypes,
    "n_nulls":  df.isna().sum(),
    "n_unique": df.nunique(),
})

,dtype,n_nulls,n_unique
record-identifier,string,0,2552
event-time,string,0,44
total-est-attendance,Int64,0,558
taxable-sales-usd,Int64,0,2552
write_time,datetime64[ns],0,305
api_invocation_time,datetime64[ns],0,109
is_deleted,boolean,0,1
